# LeetCode #146: LRU Cache

https://leetcode.com/problems/lru-cache/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| Array/List Simulation | O(n) per op | O(n) |
| HashMap + Doubly Linked List ★ | O(1) per op | O(capacity) |

## Understanding the Methods
### Brute Force (Array/List)
Use an ordered list to track usage order. On get/put, scan the list to update order. O(n) per operation.

### Optimal: HashMap + Doubly Linked List ★
HashMap for O(1) key lookup; doubly linked list to maintain usage order (most recent at head, LRU at tail).
- get: if key in map, move node to head, return value
- put: if key exists update and move to head; else create node at head; if over capacity remove tail
Both operations O(1).

## Solutions
### C#

In [ ]:
// HashMap + Doubly Linked List O(1) per operation
public class LRUCache {
    private int capacity;
    private Dictionary<int, LinkedListNode<(int key, int val)>> map = new();
    private LinkedList<(int key, int val)> list = new();

    public LRUCache(int capacity) { this.capacity = capacity; }

    public int Get(int key) {
        if (!map.ContainsKey(key)) return -1;
        var node = map[key];
        list.Remove(node);
        list.AddFirst(node);
        return node.Value.val;
    }

    public void Put(int key, int value) {
        if (map.ContainsKey(key)) {
            list.Remove(map[key]);
            map.Remove(key);
        }
        var node = list.AddFirst((key, value));
        map[key] = node;
        if (map.Count > capacity) {
            map.Remove(list.Last.Value.key);
            list.RemoveLast();
        }
    }
}

### Python

In [ ]:
# OrderedDict for O(1) LRU (Python built-in handles ordering)
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)

### Go

In [ ]:
// HashMap + Doubly Linked List O(1) per operation
import "container/list"

type LRUCache struct {
    cap  int
    m    map[int]*list.Element
    l    *list.List
}

type entry struct{ key, val int }

func Constructor(capacity int) LRUCache {
    return LRUCache{cap: capacity, m: make(map[int]*list.Element), l: list.New()}
}

func (c *LRUCache) Get(key int) int {
    if el, ok := c.m[key]; ok {
        c.l.MoveToFront(el)
        return el.Value.(*entry).val
    }
    return -1
}

func (c *LRUCache) Put(key, value int) {
    if el, ok := c.m[key]; ok {
        el.Value.(*entry).val = value
        c.l.MoveToFront(el)
        return
    }
    el := c.l.PushFront(&entry{key, value})
    c.m[key] = el
    if c.l.Len() > c.cap {
        back := c.l.Back()
        c.l.Remove(back)
        delete(c.m, back.Value.(*entry).key)
    }
}

### Rust

In [ ]:
// HashMap + IndexMap (ordered) for LRU in Rust
use std::collections::HashMap;

struct LRUCache {
    capacity: usize,
    map: HashMap<i32, i32>,
    order: std::collections::VecDeque<i32>,
}

impl LRUCache {
    fn new(capacity: i32) -> Self {
        LRUCache { capacity: capacity as usize, map: HashMap::new(), order: std::collections::VecDeque::new() }
    }
    fn get(&mut self, key: i32) -> i32 {
        if let Some(&val) = self.map.get(&key) {
            self.order.retain(|&k| k != key);
            self.order.push_back(key);
            val
        } else { -1 }
    }
    fn put(&mut self, key: i32, value: i32) {
        self.order.retain(|&k| k != key);
        self.order.push_back(key);
        self.map.insert(key, value);
        if self.map.len() > self.capacity {
            if let Some(lru) = self.order.pop_front() { self.map.remove(&lru); }
        }
    }
}

## Examples
1. **Common**: capacity=2; put(1,1),put(2,2),get(1)=1,put(3,3),get(2)=-1 (evicted)
2. **Slightly Complex**: capacity=2; put(1,1),put(2,2),put(1,10),get(1)=10,put(3,3),get(2)=-1
3. **Edge Time**: capacity=1; put(1,1),get(1)=1,put(2,2),get(1)=-1
4. **Edge Space**: capacity=10; all gets return -1 before any puts
5. **Almost-Impossible**: capacity=2; put/get interleaved to test all eviction paths